# 13 · SkewerJAX and fake_spectra: where the time goes

How long the two codes take to compute the same spectra, stage by stage, on one target: **TNG50-2** (snapshot 25,
$z = 3.0$, 1.2 billion gas cells, 349 GB), a regular **grid of $128 \times 128$ skewers along $y$** (16,384 skewers),
fake_spectra's pixels of 1 km/s, and the lines **$\mathrm{Ly}\alpha$** and **Si III 1206**. The grids $x, z = iL/n$ nest,
and notebook 14 gives the slope of every stage against the number of skewers, up to $256^2$ (65,536 skewers).
Four entries, each timed by stage on the same machine:

- **fake_spectra** (native, kernel `skewerjax-fs`, its OpenMP on every core);
- **SkewerJAX default** and **SkewerJAX matched** (every option at fake_spectra's value, notebook 12) on the **CPU**;
- **SkewerJAX default** on the **GPU**.

The stages: **setup** (the skewers), **I/O** (the snapshot reads), **cell selection** (the cells within reach of the
skewers), **gas physics** (temperatures, ion fractions), **pieces** (the crossings or chords of the cells with the
skewers) and the **$\tau$ sum**. fake_spectra searches its particles, deposits them and sums $\tau$ inside its C engine:
its selection and pieces are in its $\tau$. What each code computes, and how close they are, is in notebook 12; how the
design of SkewerJAX makes it fast, in `docs/performance.md`.

In [ ]:
import os
N_CPU = os.cpu_count()                                   # logical CPUs of this machine
os.environ["OMP_NUM_THREADS"] = str(N_CPU)               # fake_spectra's OpenMP threads (its C engine)
import skewerjax.config as config                        # imports neither JAX nor the package: the devices come first
config.configure_devices(
    n_cpu_devices=2,          # the CPU as two JAX devices: the engine overlaps two kernels on them (docs/performance.md, sec. 5)
    platforms="cuda,cpu",     # the GPU next to the CPU (config.devices("gpu") is empty without one)
)
import contextlib, io, logging, tempfile, time
logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import add_panel_letters, plot_stage_grid, use_backend, use_style
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # paper style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

CPU, GPU = config.devices("cpu"), config.devices("gpu")
jax.config.update("jax_default_device", CPU[0])          # arrays on the host by default; only the GPU entry's tau goes to the GPU
cpu_model = next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
ram_GB = next(int(l.split()[1]) for l in open("/proc/meminfo") if l.startswith("MemTotal")) / 1e6
print(f"CPU: {cpu_model}, {N_CPU} logical CPUs, {ram_GB:.0f} GB RAM; JAX CPU devices {CPU}")
print(f"GPU: {[d.device_kind for d in GPU] or 'none'}; precision {config.get_precision()} (the package's default)")

## The set-up

The skewers run along $y$ across the box: in **box mode** they share one path coordinate, so one $a(s)$, solved once in
float64 on the host with $a = a_{\rm snap}$ at the middle of the path, one frame and one pixel grid. The pixels are
fake_spectra's: $n_{\rm bins} = \lfloor v_{\max}/1\ \mathrm{km/s}\rfloor$, $dv = v_{\max}/n_{\rm bins}$,
$v_{\max} = (aH)_{\rm snap}L$.

**The streaming route** (`simulations.stream`). SkewerJAX reads the snapshot **once** for every entry: file by file, in
blocks of 2 million rows on every core, it reads the positions, masses and densities, keeps the cells within reach
($2.5\,V^{1/3}$) of some skewer (index arithmetic on the plane of the skewers, no tree), reads the other datasets of those
rows only, computes their gas physics in both configurations (the default and the matched one), and keeps a compact
record per cell. Then the skewers go in **tiles** of about 1,024 neighbours: each tile's cells give their crossings with
the skewers (the default: Voronoi cells, exact) or fake_spectra's chords (the matched mode), the contributions of every
crossing to each line, and the $\tau$ sum (`core.tau_engine`), the host preparing the next tile while the device sums the
current one.

**Protocol.** **Cold**: the snapshot is dropped from the page cache (`posix_fadvise`, nothing written) before each code
reads it, so both read the disk (the snapshot is larger than the RAM anyway). A **warm-up** on TNG50-4 compiles the same
kernels first (JAX, numba) without reading TNG50-2: its time is the JIT, reported apart. fake_spectra runs once
(`N_RUNS_FS`, 7 minutes a run), SkewerJAX `N_RUNS` times; the bars are their medians. Every result is pulled to the host,
which waits until it is ready. Everything is in float32 (the package's precision; fake_spectra's C engine works in
double).

In [ ]:
from skewerjax.core import LineBundle, SnapshotFrame
from skewerjax.simulations import ArepoHDF5Reader, stream
from skewerjax.comparison.benchmark import StageTimer
from skewerjax.comparison.scaling import MemorySampler, cache_label, cache_state, evict_page_cache, nested_grid, reference_index
from skewerjax.utils.paths import DATA_DIR as DATA   # $SKEWERJAX_DATA (docs/installation.md)

In [ ]:
lines = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
SPECIES = list(lines.species_keys)

In [ ]:
SIM, SNAP, AXIS, N_GRID = "TNG50-2", 25, 2, 128                        # along y (axis 2); N_GRID x N_GRID skewers
N_RUNS, N_RUNS_FS, TILE = 2, 1, 1024                                   # timed runs (SkewerJAX, fake_spectra); skewers per tile
reader = ArepoHDF5Reader(f"{DATA}/{SIM}", SNAP)
H = reader.header
L, AH, a_snap = H.box_cMpc, H.velfac_kms_per_cMpc, H.a                 # [cMpc], (aH)_snap [km/s per cMpc]
nbins, dv = H.pixel_grid(1.0)                                          # fake_spectra's pixels
v = (np.arange(nbins) + 0.5) * dv                                      # [km/s], over [0, v_max)
V_PERIOD = nbins * dv
SNAPDIR = reader.files[0].parent                                       # for the page cache (fincore, posix_fadvise)
print(f"{SIM}, z = {H.redshift:.3f}, L = {L:.3f} cMpc, {H.n_gas:,} gas cells in {H.n_files} files; {nbins} pixels of {dv:.6f} km/s")

In [ ]:
CPU_DEF, CPU_MAT, GPU_DEF, FS = "SkewerJAX default, CPU", "SkewerJAX matched, CPU", "SkewerJAX default, GPU", "fake_spectra"
STAGES = ("setup", "io", "selection", "physics", "pieces", "tau")
X, Z = nested_grid(N_GRID, L)                          # x, z = i L / N_GRID [cMpc], x outer: the grids of notebook 14 nest
N_SK = X.size
REF = reference_index(N_GRID)                          # the skewer at (L/4, L/4), notebook 12's
cofm = np.column_stack([X, np.zeros_like(X), Z])       # a point of each skewer [cMpc] (its y is ignored)
print(f"{N_SK:,} skewers; the reference skewer {REF} at x = {X[REF]:.3f}, z = {Z[REF]:.3f} cMpc")

## fake_spectra

`Spectra` on the same sightlines (an axis and a point in its units, ckpc/$h$) and the same pixel width, then `get_tau`
for each line. Its stages are timed from outside, without editing it: the construction of `Spectra` (**setup**), every call
of its snapshot reader `snapshot_set.get_data` (**I/O**), every call of its C engine `_do_interpolation_work` (**$\tau$**: the
particles near the sightlines, their deposition and the sum), and the rest of `get_tau` (**gas physics**: temperatures,
neutral fractions, the CLOUDY table, on every particle of every file). It reads the snapshot once per line. Its save file
goes to a temporary folder, removed afterwards, and `reload_file=True` makes it recompute every time.

In [ ]:
from fake_spectra.spectra import Spectra
cofm_fs = cofm / H.code_length_cMpc                                       # [ckpc/h]
axis = np.full(X.size, AXIS, dtype=np.int32)                              # 2 = y
fs_runs, fs_cache = [], []
for run in range(N_RUNS_FS):
    evict_page_cache(SNAPDIR)                                             # cold
    fs_cache.append(cache_state(SNAPDIR))                                 # the fraction of the snapshot in the page cache
    T = StageTimer()
    with tempfile.TemporaryDirectory(prefix="fs_timing_") as work, contextlib.redirect_stdout(io.StringIO()):
        with T.stage("setup"):
            fs = Spectra(SNAP, f"{DATA}/{SIM}", cofm_fs.copy(), axis.copy(), res=1.0, savefile="t.hdf5", savedir=work,
                         reload_file=True, quiet=True)
        with T.patched((fs.snapshot_set, "get_data", "io"), (fs, "_do_interpolation_work", "tau")):
            with T.stage("physics"):
                tau_fs = np.stack([np.asarray(fs.get_tau("H", 1, 1215)), np.asarray(fs.get_tau("Si", 3, 1206))])
    fs_runs.append({s: float(T.t.get(s, 0.0)) for s in STAGES})
    print(f"run {run}: " + ", ".join(f"{k} {t:.1f} s" for k, t in T.t.items() if t > 0) + f" (page cache {fs_cache[-1]:.0%}: {cache_label(fs_cache[-1])})")
print(f"{os.environ['OMP_NUM_THREADS']} OpenMP threads; kernel {fs.kernel_int} (0 = top hat); {fs.nbins} pixels of {fs.dvbin:.6f} km/s")
del fs

## SkewerJAX: one pass, then the tiles

The calls, every option written. `stream.stream_cells` makes the pass with both physics configurations; then
`stream.tiles_optical_depth` computes each entry from the same store: the default (Voronoi crossings, the integrated
profile, the exact frame) on the CPU and, from the same host work, on the GPU; the matched mode (fake_spectra's chords
with its weights, its frame, its per-cell early stop `tautail` and pixel average) on the CPU.

The stages of the pass overlap on its threads (a block is read while another one is selected or converted), so its wall
time is split by each stage's share of the threads' time: **I/O** (the reads, with the pass's set-up and the assembly of
the store), **selection** (the reach test and the cell-to-tile lists), **gas physics** (each entry its own configuration;
the GPU entry the default's). **Pieces** is the time the device waits for the host to prepare a tile (the crossings or
chords and their contributions, on three host threads while the device sums), **$\tau$** the engine's time. An entry's
total is the pass plus its tiles; the pass is shared, and run once.

In [ ]:
DEFAULT_CELLS = dict(mass_unit="amu", T_constants="codata", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")
MATCHED_CELLS = dict(mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra",
                     cloudy_interpolation="cubic")
DEFAULT_TAU = dict(pieces="crossings", profile="integrated", detuning="exact", voigt="harris", hubble_flow="exact", a_eff=None,
                   b_constants="codata", velocity_period=None, tautail=None, pixel_average=False)
MATCHED_TAU = dict(pieces="chords", chord_weight=1 / 8, profile="midpoint", detuning="linear", voigt="harris", hubble_flow="linear",
                   aH_ref=AH, a_eff=a_snap, b_constants="fake_spectra", velocity_period=V_PERIOD, tautail=1e-7, pixel_average=True)


def skewerjax_run(reader, cofm, memory=None):
    """The three SkewerJAX entries on the skewers ``cofm``: ({entry: tau}, {entry: {stage: s}}, the pass's info)."""
    t0 = time.perf_counter()
    paths, frame = stream.box_skewers(cofm, AXIS, reader.header.box_cMpc, reader.header.cosmology(), reader.header.a)
    tile_of = stream.tile_skewers(paths, TILE)                                    # squares of the skewers' plane
    snap_frame = SnapshotFrame(reader.header.a, reader.header.velfac_kms_per_cMpc, path_length=np.full(len(paths), reader.header.box_cMpc))
    setup = time.perf_counter() - t0
    store, pinfo = stream.stream_cells(reader, paths, tile_of, SPECIES, {"default": DEFAULT_CELLS, "matched": MATCHED_CELLS})
    share = pinfo["timings"]                                                      # each stage's share of the pass's wall time
    wall = pinfo["pass_s"] + pinfo["setup_s"] + pinfo["assemble_s"]
    io_s = wall - share["reach"] - share["tiles"] - share["physics:default"] - share["physics:matched"]
    sel_s = share["reach"] + share["tiles"]
    taus, stages = {}, {}
    runs = [(CPU_DEF, "default", DEFAULT_TAU, frame, CPU), (CPU_MAT, "matched", MATCHED_TAU, snap_frame, CPU)]
    if GPU:
        runs.append((GPU_DEF, "default", DEFAULT_TAU, frame, GPU))   # the same host work, the sum on the GPU
    for e, cfg, opts, fr, devs in runs:
        out, tinfo = stream.tiles_optical_depth(store, cfg, paths, tile_of, fr, v, lines, engines={"tau": devs}, **opts)
        rows = tinfo["rows"]
        taus[e] = out["tau"]
        stages[e] = dict(setup=setup, io=io_s, selection=sel_s, physics=share[f"physics:{cfg}"],
                         pieces=sum(r["wait_s"] for r in rows), tau=sum(r["tau_s"]["tau"] for r in rows))
    return taus, stages, pinfo

## The runs

The warm-up runs the same calls on TNG50-4 (the JIT compilation of every kernel); then `N_RUNS` cold runs on TNG50-2. The
$\tau$ of every skewer is kept from the last run for the checks below.

In [ ]:
t = time.perf_counter()
warm = ArepoHDF5Reader(f"{DATA}/TNG50-4", SNAP)
Xw, Zw = nested_grid(32, warm.header.box_cMpc)
skewerjax_run(warm, np.column_stack([Xw, 0 * Xw, Zw]))                 # one tile of TNG50-4 skewers: compiles every kernel
JIT = time.perf_counter() - t
sj_runs, sj_cache, mem = [], [], MemorySampler()
for run in range(N_RUNS):
    evict_page_cache(SNAPDIR)                                          # cold
    sj_cache.append(cache_state(SNAPDIR))
    mem.reset()
    TAU, st, pinfo = skewerjax_run(reader, cofm, mem)
    sj_runs.append(st)
    print(f"run {run}: " + ", ".join(f"{e} {sum(s.values()):.0f} s" for e, s in st.items())
          + f"; the pass {pinfo['pass_s']:.0f} s over {pinfo['n_needed']:,} cells ({pinfo['fraction']:.0%}), peak memory {mem.read()['rss_GB']:.0f} GB")
print(f"warm-up (JIT compilation of every kernel, TNG50-4): {JIT:.0f} s")

## Fairness and correctness

The four entries compute the same spectra of the same two lines on the same pixels, in float32 (fake_spectra in double).
The checks: the matched mode against fake_spectra (rms and max $\Delta F$ over every pixel of every skewer), the GPU against
the CPU for the default (to float32 rounding), and the default against the matched mode (the differences of physics of
notebook 12). **The page cache:** the fraction of the snapshot in the page cache before each code reads it is printed
below (the protocol drops it: both codes read the disk).

In [ ]:
F = lambda t: np.exp(-np.asarray(t, dtype=np.float64))
rms = lambda a, b: float(np.sqrt(np.mean((F(a) - F(b)) ** 2)))
mx = lambda a, b: float(np.max(np.abs(F(a) - F(b))))
for j, line in enumerate(lines.lines):
    print(f"{line.id:11s} matched (CPU) vs fake_spectra: rms dF {rms(TAU[CPU_MAT][j], tau_fs[j]):.1e}, max {mx(TAU[CPU_MAT][j], tau_fs[j]):.1e}"
          f" | default vs matched: rms dF {rms(TAU[CPU_DEF][j], TAU[CPU_MAT][j]):.1e}")
    if GPU:
        print(f"{'':11s} default, GPU vs CPU: rms dF {rms(TAU[GPU_DEF][j], TAU[CPU_DEF][j]):.1e}, max {mx(TAU[GPU_DEF][j], TAU[CPU_DEF][j]):.1e}")
print("page cache before the reads: fake_spectra " + ", ".join(f"{c:.0%}" for c in fs_cache) + "; SkewerJAX " + ", ".join(f"{c:.0%}" for c in sj_cache))
med = {FS: {s: float(np.median([r[s] for r in fs_runs])) for s in STAGES}}
med.update({e: {s: float(np.median([r[e][s] for r in sj_runs])) for s in STAGES} for e in sj_runs[0]})
print()
print(f"{'':24s}" + "".join(f"{s:>11s}" for s in STAGES) + f"{'total':>11s}")
for e, st in med.items():
    print(f"{e:24s}" + "".join(f"{st[s]:11.1f}" for s in STAGES) + f"{sum(st.values()):11.1f}")

## Where the time goes

The median wall time of one run: (a) split into its stages, with the total and the JIT compilation of the warm-up (not
in the bars) after each bar; (b) each stage's share; (c) the totals on a log axis with the speed-up over fake_spectra.

In [ ]:
order = [FS, CPU_DEF, CPU_MAT] + ([GPU_DEF] if GPU else [])
fig, axs = plot_stage_grid([{e: med[e] for e in order}], jit=[{e: (JIT if e != FS else 0.0) for e in order}], figsize=(22, 6.5))
add_panel_letters(axs[0])
fig.suptitle("SkewerJAX's time goes to the one read of the snapshot and to its τ sum", fontsize=21)
plt.show()
totals = {e: sum(med[e].values()) for e in order}
for e in order:
    print(f"{e:24s} {totals[e]:8.1f} s" + ("" if e == FS else f": {totals[FS] / totals[e]:5.2f}x fake_spectra's speed"))

## Discussion

**SkewerJAX is 3.7 times faster than fake_spectra on the CPU, 4.7 times on the GPU and 2.7 times in the matched mode** (TNG50-2,
$128^2$ skewers, cold, the median of two runs: fake_spectra 494 s; SkewerJAX default CPU 135 s, matched 182 s, GPU 104
s). Before this work (commit `01d9900`) the same target took SkewerJAX 2,989 s on the CPU and 1,265 s on the GPU.

- **fake_spectra** spends 166 s reading the snapshot, 139 s on the gas physics of all its 1.2 billion cells, and 189 s
  in its C engine (11.5 ms per skewer): its fixed cost, ~310 s, is set by the snapshot, not by the skewers.
- **SkewerJAX** reads the snapshot once for its three entries (the pass, ~72 s: I/O 48 s, the cells within reach of the
  skewers 17 s, the gas physics of those 23% of the cells 7 s) and then spends 57 s on the $\tau$ sum on the CPU (3.5 ms per
  skewer), 28 s on the GPU and 105 s in the matched mode, which keeps fake_spectra's per-cell walk. The crossings and
  contributions of the next tile are prepared while the current one is summed: their wait is 4–5 s.
- **The same spectra.** The matched mode reproduces fake_spectra to rms $\Delta F$ 1.9e-7 ($\mathrm{Ly}\alpha$) and 2.6e-8 (Si III) over every
  pixel of every skewer; the GPU agrees with the CPU to 4.1e-7. The default differs from both by its own choices
  (notebook 12): the Voronoi cells instead of fake_spectra's spheres, its H I model, the exact frame.

At $256^2$ skewers (notebook 14) fake_spectra takes 1,069 s and SkewerJAX 386 s (CPU), 273 s (GPU) and 562 s (matched),
with a peak memory of 83 GB; at 16 skewers, 308 s against 26 s. `docs/performance.md` explains the design: the cell
crossings with the profile integrated over each, the windowed engine, and the one pass over the snapshot.
